# SUMO ML Pipeline - Traffic Flow Prediction

This notebook demonstrates the complete ML pipeline for traffic flow prediction:
1. Generate network topology
2. Run SUMO simulation (or synthetic fallback)
3. Feature engineering
4. Train multiple regression models
5. Evaluate and compare models
6. Export best model (joblib + ONNX)
7. Make predictions via API

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from traffic_control import (
    generate_manhattan_example,
    generate_four_junction_example,
    run_ml_pipeline,
    predict_flows,
    TrafficFlowRegressor,
    SUMODataGenerator,
    create_features,
    prepare_training_data,
)

## 1. Generate Network Topology

In [ ]:
# Generate a Manhattan-style grid network
network = generate_manhattan_example()

print(f"Junctions: {len(network.junctions)}")
print(f"Roads: {len(network.roads)}")

# Show network structure
for j in network.junctions[:5]:
    print(f"  Junction {j.id}: pos={j.position}, ext_flow={j.external_flow}")
for r in network.roads[:5]:
    print(f"  Road {r.id}: {r.source} -> {r.target}, cap={r.capacity}")

## 2. Generate Training Data (SUMO or Synthetic)

In [ ]:
# Create data generator
generator = SUMODataGenerator()

# Generate dataset (will use SUMO if available, otherwise synthetic fallback)
print("Generating traffic data...")
df = generator.generate_dataset(network, name="manhattan", duration=3600)

print(f"Dataset shape: {df.shape}")
print(f"Time range: {df['time'].min()} - {df['time'].max()}")
print(f"Unique edges: {df['edge_id'].nunique()}")
print(df.head())

## 3. Feature Engineering

In [ ]:
# Create features for ML
df_features = create_features(df, network)

print(f"Features shape: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")
print(df_features.head())

## 4. Prepare Training Data with Lookback Window

In [ ]:
# Prepare training data with 4-step lookback (15-min intervals = 1 hour history)
X, y, feature_names = prepare_training_data(
    df_features,
    target="flow",
    lookback=4
)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Feature names ({len(feature_names)}): {feature_names[:10]}...")

# Time-based split (80/20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")

## 5. Train Multiple Models

In [ ]:
# Train multiple model types
model_types = ["linear", "ridge", "random_forest", "xgboost"]

models = {}
for mt in model_types:
    try:
        reg = TrafficFlowRegressor(model_type=mt)
        reg.fit(X_train, y_train, feature_names)
        models[mt] = reg
        print(f"Trained {mt}")
    except Exception as e:
        print(f"Failed to train {mt}: {e}")

## 6. Evaluate and Compare Models

In [ ]:
# Evaluate all models on test set
results = []
for name, model in models.items():
    metrics = model.evaluate(X_test, y_test)
    results.append({
        "model": name,
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "mape": metrics["mape"],
    })
    print(f"{name}: MAE={metrics['mae']:.4f}, RMSE={metrics['rmse']:.4f}, MAPE={metrics['mape']:.4f}")

In [ ]:
# Visualize model comparison
results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, metric in enumerate(["mae", "rmse", "mape"]):
    ax = axes[i]
    bars = ax.bar(results_df["model"], results_df[metric], color=sns.color_palette("viridis", len(results_df)))
    ax.set_title(metric.upper())
    ax.set_ylabel(metric.upper())
    ax.tick_params(axis='x', rotation=45)
    # Add value labels
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha='center', va='bottom')

plt.suptitle("Model Comparison on Test Set")
plt.tight_layout()
plt.show()

## 7. Cross-Validation (Time Series Split)

In [ ]:
# Cross-validate best model
best_model_name = min(results, key=lambda x: x["mae"]) ["model"]
best_model = models[best_model_name]

cv_results = best_model.cross_validate(X, y, n_splits=5)
print(f"Cross-validation results for {best_model_name}:")
for k, v in cv_results.items():
    print(f"  {k}: {v:.4f}")

## 8. Feature Importance

In [ ]:
# Show feature importance for tree-based models
for name, model in models.items():
    importance = model.get_feature_importance()
    if importance:
        print(f"\n{name} - Top 10 Features:")
        sorted_imp = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
        for feat, imp in sorted_imp:
            print(f"  {feat}: {imp:.4f}")

## 9. Save Models (Joblib + ONNX)

In [ ]:
# Save all models with ONNX export
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    joblib_path = model_dir / f"{name}.joblib"
    model.save(joblib_path, export_onnx=True)
    print(f"Saved {name} to {joblib_path}")

## 10. Load Model and Make Predictions

In [ ]:
# Load best model and make predictions
loaded_model = TrafficFlowRegressor.load(model_dir / f"{best_model_name}.joblib")

# Predict on test set
preds = loaded_model.predict(X_test)

# Compare predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Time series comparison (first edge)
edge_0_mask = X_test[:, -len(feature_names)//4:]  # Approximate - last timestep features
sample_idx = np.arange(min(100, len(y_test)))
axes[0].plot(sample_idx, y_test[sample_idx], 'o-', label='Actual', alpha=0.7)
axes[0].plot(sample_idx, preds[sample_idx], 's-', label='Predicted', alpha=0.7)
axes[0].set_xlabel("Sample")
axes[0].set_ylabel("Flow (veh/h)")
axes[0].set_title("Predictions vs Actual (First 100 Samples)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot
axes[1].scatter(y_test, preds, alpha=0.5, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect')
axes[1].set_xlabel("Actual Flow")
axes[1].set_ylabel("Predicted Flow")
axes[1].set_title("Predicted vs Actual")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Make API Predictions

In [ ]:
# Example: Predict flow for specific features
sample_features = {
    "hour": 8,
    "time_of_day": np.sin(2 * np.pi * 8 / 24),
    "length": 200.0,
    "capacity": 1000.0,
    "cost": 1.0,
    "utilization": 0.3,
    "travel_time": 120.0,
}

# Create feature vector matching training format
X_sample = np.array([[sample_features.get(f, 0) for f in loaded_model.features]])
prediction = loaded_model.predict(X_sample)

print(f"Predicted flow: {prediction[0]:.1f} veh/h")

## 12. Full Pipeline in One Call

In [ ]:
# Run the complete pipeline with one function
print("Running full ML pipeline...")
models = run_ml_pipeline(
    network,
    output_dir="../models",
    model_types=["linear", "ridge", "random_forest", "xgboost"],
    duration=3600,
    lookback=4,
)

print("Pipeline complete!")